# Athéon — production locale T4, sans Google Drive

**6 scènes / 19 plans / 2672 images / 540×960 / Candidate 48.** Ce notebook utilise `/content/atheon-cinematic-local-01`, qui disparaît si Colab réinitialise son environnement. Télécharge **chaque archive de scène** avant de quitter Colab. Pour reprendre, relance 1 et 2, puis réimporte **toutes** les archives sauvegardées en 3. Pas de benchmark, pas de GitHub Actions. Les téléchargements et téléversements volumineux peuvent être difficiles sur Chrome Android.


In [ ]:
# @title 1. Installer le moteur Athéon sans Drive (T4)
from google.colab import output
from pathlib import Path
import hashlib, json, os, subprocess, sys, tarfile, urllib.request
from IPython.display import display, Video, FileLink
output.enable_custom_widget_manager()
print('Mode local : aucun montage Google Drive')
projet_a_reprendre = '' # Ne pas utiliser : restaurer les archives avec la cellule 3
prepared_pilot_only = True
DRIVE_ROOT = Path('/content')
REPO = Path('/content/Destiny-Explained-Studio')
BRANCH = 'feat/atheon-performance-cinematic-direction'
def run(args, **kwargs):
    subprocess.run([str(x) for x in args], check=True, **kwargs)
if not REPO.exists():
    run(['git','clone','--depth','1','--single-branch','--branch',BRANCH,'https://github.com/Alexanst54/Destiny-Explained-Studio.git',REPO])
else:
    actual = subprocess.check_output(['git','-C',REPO,'branch','--show-current'],text=True).strip()
    assert actual in [BRANCH,''], 'Dossier sur une autre branche : utilisez un runtime neuf.'
print('Source épinglée dans ce runtime :',subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())
run(['apt-get','update','-qq'])
asound='libasound2t64' if subprocess.run(['apt-cache','show','libasound2t64'],capture_output=True).returncode==0 else 'libasound2'
run(['apt-get','install','-y','-qq','ffmpeg','espeak-ng','python3-venv','libxi6','libxrender1','libxkbcommon0','libsm6','libice6','libgl1','libegl1','libnss3','libgbm1','fonts-dejavu-core',asound])
def install_archive(url, checksum_url, name, destination):
    archive=Path('/content')/name
    urllib.request.urlretrieve(url,str(archive))
    checks=urllib.request.urlopen(checksum_url).read().decode()
    digest=hashlib.file_digest(archive.open('rb'),'sha256').hexdigest()
    assert any(digest in line and name in line for line in checks.splitlines()), 'Téléchargement invalide : SHA-256 incorrect.'
    # Official, SHA-verified archives only.
    run(['tar','-xf',archive,'-C',destination])
    archive.unlink()
NODE_DIR=Path('/content/node-v22.23.4-linux-x64')
if not NODE_DIR.exists():
    base='https://nodejs.org/dist/v22.23.4/'
    install_archive(base+'node-v22.23.4-linux-x64.tar.xz',base+'SHASUMS256.txt','node-v22.23.4-linux-x64.tar.xz','/content')
os.environ['PATH']=str(NODE_DIR/'bin')+':'+os.environ['PATH']
os.environ['HF_HOME']='/content/destiny-model-cache/huggingface'
os.environ['DESTINY_MODEL_CACHE']='/content/destiny-model-cache'
NODE=NODE_DIR/'bin/node'
BLENDER=Path('/content/blender-4.3.2-linux-x64/blender')
if not BLENDER.exists():
    base='https://download.blender.org/release/Blender4.3/'
    install_archive(base+'blender-4.3.2-linux-x64.tar.xz',base+'blender-4.3.2.sha256','blender-4.3.2-linux-x64.tar.xz','/content')
VENV=Path('/content/destiny-studio-venv')
if not VENV.exists(): run([sys.executable,'-m','venv',VENV])
PYTHON=VENV/'bin/python'
if not prepared_pilot_only:
    run([PYTHON,'-m','pip','install','--no-cache-dir','--index-url','https://download.pytorch.org/whl/cpu','torch==2.6.0','torchaudio==2.6.0'])
    run([PYTHON,'-m','pip','install','--no-cache-dir','-r',REPO/'remotion/studio/requirements.txt'])
else:
    print('Prepared Atheon pilot only: original six WAVs will be reused; Torch/Kokoro model installation is not needed. To use arbitrary Studio narration later, uncheck this option and rerun installation.')
run(['npm','ci'],cwd=REPO/'remotion')
run([NODE,REPO/'remotion/node_modules/@remotion/cli/remotion-cli.js','browser','ensure'],cwd=REPO/'remotion')
try: run(['nvidia-smi'])
except (FileNotFoundError,subprocess.CalledProcessError): print('ATTENTION : GPU NVIDIA indisponible. Blender utilisera le CPU si nécessaire.')
sys.path.insert(0,str(REPO/'remotion/studio'))
from colab_ui import form
print('Installation terminée. Aucun appel à GitHub Actions.')


In [ ]:
# @title 2. Préparer le projet local (aucun rendu)
from pathlib import Path
import subprocess, json, shutil, zipfile, hashlib
from google.colab import files
FOLDER=Path('/content/atheon-cinematic-local-01')
assert FOLDER.resolve()==FOLDER and str(FOLDER).startswith('/content/'), 'Dossier local uniquement'
assert shutil.which('nvidia-smi'), 'Sélectionne un environnement GPU T4 dans Colab'
CINEMATIC_ARGS=[str(PYTHON),str(REPO/'remotion/studio/cinematic_atheon.py')]
CINEMATIC_OPTIONS=[str(FOLDER),'--scratch','/content/atheon-cinematic-scratch','--width','540','--height','960','--blender',str(BLENDER),'--node',str(NODE)]
def cinematic_action(action,extra=()):
    subprocess.run(CINEMATIC_ARGS+[action]+CINEMATIC_OPTIONS+list(extra),check=True)
cinematic_action('prepare')
print('Projet prêt :',FOLDER)
print('Espace disponible (Go) :',round(shutil.disk_usage('/content').free/1e9,2))


In [ ]:
# @title 3. Restaurer les archives ZIP de scènes (facultatif, après reconnexion)
# Après un redémarrage, relance les cellules 1 et 2, puis celle-ci pour CHAQUE archive sauvegardée.
# Chrome Android peut limiter les envois volumineux. N'efface pas les archives du téléphone.
import io, os
restaurer_archives = False # @param {type:"boolean"}
if restaurer_archives:
    incoming=files.upload()
    for name,blob in incoming.items():
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            names=z.namelist()
            assert 'project.json' in names, 'Archive de projet incomplète'
            assert z.testzip() is None, 'ZIP corrompu'
            old=json.loads(z.read('project.json'))
            current=json.loads((FOLDER/'project.json').read_text())
            assert old['fingerprint']==current['fingerprint'], 'Projet/code/paramètres incompatibles : ne pas mélanger les archives'
            for member in z.infolist():
                rel=Path(member.filename)
                assert not rel.is_absolute() and '..' not in rel.parts and not member.is_dir() or member.is_dir() and not rel.is_absolute() and '..' not in rel.parts, 'Chemin non sûr'
                if member.is_dir(): continue
                assert not member.filename.startswith('/') and not member.filename.endswith('.lock'), 'Archive invalide'
                target=(FOLDER/rel).resolve()
                assert target.is_relative_to(FOLDER.resolve()), 'Sortie du dossier projet'
                target.parent.mkdir(parents=True,exist_ok=True)
                if target.exists():
                    with z.open(member) as src:
                        h=hashlib.sha256()
                        for chunk in iter(lambda:src.read(1024*1024),b''):h.update(chunk)
                    assert hashlib.file_digest(target.open('rb'),'sha256').hexdigest()==h.hexdigest(), 'Conflit de fichiers : '+member.filename
                else:
                    with z.open(member) as src, target.open('wb') as dst: shutil.copyfileobj(src,dst)
        print('Restauré :',name)
else:
    print('Première session : aucune archive à restaurer.')


In [ ]:
# @title 4. Rendre UNE scène sur T4 avec Candidate 48
scene_to_render = '01-hook' # @param ['01-hook','02-identity','03-vault','04-time','05-guardians','06-mystery']
# Production explicitement approuvée : pas de previews/benchmarks supplémentaires.
cinematic_action('render',['--production-approved-no-previews','--scene',scene_to_render])
print('Scène terminée :',scene_to_render)


In [ ]:
# @title 5. Sauvegarder la scène en ZIP et télécharger sur Android
# Exécute cette cellule IMMÉDIATEMENT après chaque scène.
scene_to_save = '01-hook' # @param ['01-hook','02-identity','03-vault','04-time','05-guardians','06-mystery']
from pathlib import Path
import zipfile
scene_dir=FOLDER/'public/scenes'/scene_to_save/'frames'
assert scene_dir.exists() and list(scene_dir.rglob('profile.json')), 'Aucune image validée pour cette scène'
backup=Path('/content')/('atheon-backup-'+scene_to_save+'.zip')
files_to_pack=[FOLDER/'project.json',FOLDER/'subtitles.en.srt']
files_to_pack+=list((FOLDER/'inputs').rglob('*'))
files_to_pack+=list((FOLDER/'scenes').rglob('*'))
files_to_pack+=list((FOLDER/'public/scenes').glob('*/speech.wav'))
files_to_pack+=list(scene_dir.rglob('*'))
files_to_pack=[p for p in files_to_pack if p.is_file() and p.name!='episode.lock']
with zipfile.ZipFile(backup,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    for p in sorted(set(files_to_pack)):
        z.write(p,p.relative_to(FOLDER).as_posix())
with zipfile.ZipFile(backup) as z:
    assert z.testzip() is None, 'Archive corrompue'
print('Archive :',backup,'taille Go :',round(backup.stat().st_size/1e9,3))
files.download(str(backup))
print('Vérifie le téléchargement sur ton téléphone AVANT de quitter la session.')


In [ ]:
# @title 6. Assembler le MP4 quand les SIX scènes sont rendues ou restaurées
assembler_episode = False # @param {type:"boolean"}
assert assembler_episode, 'Active cette option seulement après avoir sauvegardé les six scènes.'
cinematic_action('assemble')
cinematic_action('validate')
video=FOLDER/'atheon-episode-01-cinematic.mp4'
proof=FOLDER/'atheon-episode-01-cinematic.validation.json'
print('MP4 validé :',video,round(video.stat().st_size/1e6,1),'Mo')
files.download(str(video))
files.download(str(proof))
